In [4]:
!pip -q install streamlit
!pip -q install google-generativeai
!pip -q install pymupdf
!pip -q install reportlab
!pip -q install plotly
!pip -q install python-dotenv
!pip -q install pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 60.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 29.9 MB/s eta 0:00:00


In [5]:
import os

folders = [
    "AI-Interview-Assistant",
    "AI-Interview-Assistant/uploads",
    "AI-Interview-Assistant/reports",
    "AI-Interview-Assistant/assets"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("Project Created")

Project Created


In [ ]:
%%writefile AI-Interview-Assistant/gemini.py

import google.generativeai as genai

import os
API_KEY = os.getenv("GEMINI_API_KEY")
genai.configure(api_key=API_KEY)

model = genai.GenerativeModel("gemini-2.5-flash")

def ask_ai(prompt):

    response = model.generate_content(prompt)

    return response.text

Writing AI-Interview-Assistant/gemini.py


In [7]:
import sys
sys.path.append("/content/AI-Interview-Assistant")


In [8]:
from gemini import ask_ai

print(ask_ai("Say hello in one sentence."))

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Hello there!


In [9]:
!pip -q install pymupdf

In [10]:
%%writefile resume_parser.py

import fitz

def extract_text(pdf_path):
    doc = fitz.open(pdf_path)

    text = ""

    for page in doc:
        text += page.get_text()

    return text

Writing resume_parser.py


In [11]:
from google.colab import files

uploaded = files.upload()

Saving resume.pdf to resume.pdf


In [12]:
from resume_parser import extract_text

pdf = list(uploaded.keys())[0]

resume = extract_text(pdf)

print(resume[:3000])

Maheshwar Kuchana
London •
maheshwar.kuchana@gmail.com • linkedin.com/in/maheshwarkuchana
Senior Machine Learning Engineer
Senior Machine Learning Engineer with 5+ years of experience in designing, developing, and deploying production-grade AI &
GenAI systems at scale. Expertise in leveraging AI frameworks, cloud technologies (Azure, AWS), and MLOps practices for robust
product lifecycle management. Proven track record of building end-to-end AI products, optimizing business decisions, and
automating processes through cutting-edge AI techniques across the retail, healthcare, ǜnance, insurance and banking sectors.
Adept at working in cross-functional teams to deliver high-impact AI solutions that drive signiǜcant business value.
SKILLS
Gen AI: Large  Language  Models  ( LLM) , LangChain, LangGraph, AgentOps, Multi- Agentic  Systems, MCP
Software Engineering: Kubernetes, Terraform, Docker, Amazon  Web  Services, Microsoft  Azure, Azure  Data  Factory, Azure  
Pipelines, Databricks, Delta 

In [13]:
from gemini import ask_ai

prompt = f"""
You are an ATS system.

Extract

1. Skills
2. Education
3. Projects
4. Experience

Resume

{resume}
"""

print(ask_ai(prompt))

Here's the extracted information:

## 1. Skills

**Gen AI:** Large Language Models (LLM), LangChain, LangGraph, AgentOps, Multi-Agentic Systems, MCP
**Software Engineering:** Kubernetes, Terraform, Docker, Amazon Web Services (AWS), Microsoft Azure, Azure Data Factory, Azure Pipelines, Databricks, Delta Lake, MLFlow, MLOps, Python, PySpark, GitHub Actions, GitLab Pipelines, FastAPI, Flask, Data Engineering, Event-Driven Architecture, Software Architecture, Design Patterns, Testing, Power BI, SQL, NoSQL DB, Vector DB
**ML & Data Science:** PyTorch, TensorFlow, Computer Vision, CNN, Self-Supervised Learning, Medical Imaging, MONAI, Feature Engineering, Statistical analysis, Hypothesis testing, A/B testing, Autoencoders, Feature Store

## 2. Education

**King's College London**
*   Master of Research in Healthcare Technologies in Artificial Intelligence
*   GPA: Distinction
*   Sep 2021 - Sep 2022

## 3. Projects

*   **Quantitative Imaging of the Shared Placenta in Twin Pregnancies:** Bu

from google.colab import files

uploaded = files.upload()

pdf = next(iter(uploaded))

from resume_parser import extract_text

resume = extract_text(pdf)

print("Loaded:", pdf)
print(resume[:500])

In [15]:
%%writefile ats.py

import re

def clean(text):
    return text.lower().strip()


def keyword_match(resume, jd):

    resume_words = set(re.findall(r"\w+", clean(resume)))
    jd_words = set(re.findall(r"\w+", clean(jd)))

    matched = resume_words.intersection(jd_words)

    score = len(matched) / len(jd_words) * 100

    return round(score,2), matched

Writing ats.py


In [16]:
from google.colab import files

uploaded = files.upload()

jd_file = next(iter(uploaded))

with open(jd_file) as f:
    job_description = f.read()

print(job_description)

Saving job.txt to job (1).txt
Machine Learning Intern

Requirements

Python
Machine Learning
TensorFlow
PyTorch
SQL
Git
Docker
Communication
Problem Solving
Deep Learning

Preferred

AWS
Azure
LangChain


In [17]:
from ats import keyword_match

score, matched = keyword_match(
    resume,
    job_description
)

print(score)

print(matched)

66.67
{'sql', 'python', 'machine', 'requirements', 'aws', 'pytorch', 'azure', 'tensorflow', 'learning', 'deep', 'langchain', 'docker'}


In [ ]:
import google.generativeai as genai
import json

import os
API_KEY = os.getenv("GEMINI_API_KEY")

genai.configure(api_key=API_KEY)

model = genai.GenerativeModel("gemini-2.5-flash")


def ask_ai(prompt):

    response = model.generate_content(prompt)

    return response.text


def ask_json(prompt):

    response = model.generate_content(prompt)

    text = response.text

    # remove markdown if Gemini returns ```json
    text = text.replace("```json", "")
    text = text.replace("```", "")

    return json.loads(text)

In [19]:
%%writefile resume_to_json.py

from gemini import ask_json

def parse_resume(resume):

    prompt = f"""
You are an ATS parser.

Return ONLY valid JSON.

Schema:

{{
"name":"",
"email":"",
"phone":"",
"skills":[],
"education":[],
"experience":[],
"projects":[]
}}

Resume:

{resume}
"""

    return ask_json(prompt)

Writing resume_to_json.py


In [ ]:
%%writefile gemini.py

import google.generativeai as genai
import json

import os
API_KEY = os.getenv("GEMINI_API_KEY")

genai.configure(api_key=API_KEY)

model = genai.GenerativeModel("gemini-2.5-flash")


def ask_ai(prompt):
    response = model.generate_content(prompt)
    return response.text


def ask_json(prompt):
    response = model.generate_content(prompt)

    text = response.text

    # Remove markdown if Gemini returns ```json ... ```
    text = text.replace("```json", "")
    text = text.replace("```", "")

    return json.loads(text)

Writing gemini.py


In [21]:
!cat /content/AI_interview_assistant/gemini.py

cat: /content/AI_interview_assistant/gemini.py: No such file or directory


In [22]:
!cat /content/gemini.py


import google.generativeai as genai
import json

API_KEY = "AQ.Ab8RN6Ly9_7uEcSpUwPJk67AOOCS6quPVdhsT1Dai-bCaqr_Tg"

genai.configure(api_key=API_KEY)

model = genai.GenerativeModel("gemini-2.5-flash")


def ask_ai(prompt):
    response = model.generate_content(prompt)
    return response.text


def ask_json(prompt):
    response = model.generate_content(prompt)

    text = response.text

    # Remove markdown if Gemini returns ```json ... ```
    text = text.replace("```json", "")
    text = text.replace("```", "")

    return json.loads(text)


In [30]:
import gemini

print(gemini.__file__)
print(dir(gemini))

/content/AI-Interview-Assistant/gemini.py
['API_KEY', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'ask_ai', 'genai', 'model']


In [ ]:
%%writefile gemini.py

import google.generativeai as genai
import json

import os
API_KEY = os.getenv("GEMINI_API_KEY")

genai.configure(api_key=API_KEY)

model = genai.GenerativeModel("gemini-2.5-flash")


def ask_ai(prompt):
    response = model.generate_content(prompt)
    return response.text


def ask_json(prompt):
    response = model.generate_content(prompt)

    text = response.text
    text = text.replace("```json", "")
    text = text.replace("```", "")

    return json.loads(text)

Overwriting gemini.py


In [34]:
import importlib
import gemini

importlib.reload(gemini)

print(dir(gemini))

['API_KEY', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'ask_ai', 'ask_json', 'genai', 'json', 'model']


In [35]:
%%writefile ats.py
from gemini import ask_json

def ats_score(resume_json, job_description):
    prompt = f"""
You are an ATS system.

Compare the resume with the job description.

Return ONLY valid JSON.

{{
  "score": 0,
  "matched_skills": [],
  "missing_skills": [],
  "strengths": [],
  "weaknesses": [],
  "suggestions": []
}}

Resume:
{resume_json}

Job Description:
{job_description}
"""

    return ask_json(prompt)

Overwriting ats.py


In [37]:
from resume_parser import extract_text
from resume_to_json import parse_resume
import json

resume = extract_text("resume.pdf")
resume_json = parse_resume(resume)

print(json.dumps(resume_json, indent=4))

{
    "name": "Maheshwar Kuchana",
    "email": "maheshwar.kuchana@gmail.com",
    "phone": "",
    "skills": [
        "Large Language Models (LLM)",
        "LangChain",
        "LangGraph",
        "AgentOps",
        "Multi-Agentic Systems",
        "MCP",
        "Kubernetes",
        "Terraform",
        "Docker",
        "Amazon Web Services",
        "Microsoft Azure",
        "Azure Data Factory",
        "Azure Pipelines",
        "Databricks",
        "Delta Lake",
        "MLFlow",
        "MLOps",
        "Python",
        "PySpark",
        "GitHub Actions",
        "GitLab Pipelines",
        "FastAPI",
        "Flask",
        "Data Engineering",
        "Event-Driven Architecture",
        "Software Architecture",
        "Design Patterns",
        "Testing",
        "Power BI",
        "SQL",
        "NoSQL DB",
        "Vector DB",
        "PyTorch",
        "TensorFlow",
        "Computer Vision",
        "CNN",
        "Self-Supervised Learning",
        "Medical I

In [38]:
!cat /content/ats.py

from gemini import ask_json

def ats_score(resume_json, job_description):
    prompt = f"""
You are an ATS system.

Compare the resume with the job description.

Return ONLY valid JSON.

{{
  "score": 0,
  "matched_skills": [],
  "missing_skills": [],
  "strengths": [],
  "weaknesses": [],
  "suggestions": []
}}

Resume:
{resume_json}

Job Description:
{job_description}
"""

    return ask_json(prompt)


In [41]:
import importlib
import ats

importlib.reload(ats)

print(dir(ats))

['__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'ask_json', 'ats_score', 'clean', 'keyword_match', 're']


In [42]:
from ats import ats_score
import json

job = open("job.txt").read()

result = ats_score(resume_json, job)

print(json.dumps(result, indent=4))

{
    "score": 100,
    "matched_skills": [
        "Python",
        "Machine Learning",
        "TensorFlow",
        "PyTorch",
        "SQL",
        "Git",
        "Docker",
        "Communication",
        "Problem Solving",
        "Deep Learning",
        "AWS",
        "Azure",
        "LangChain"
    ],
    "missing_skills": [],
    "strengths": [
        "Comprehensive expertise in Machine Learning, Deep Learning, MLOps, and cloud platforms (AWS, Azure, Kubernetes).",
        "Extensive practical experience in developing, deploying, and maintaining production-grade AI/ML systems, including multi-agentic architectures using LangChain/LangGraph.",
        "Proven ability to deliver significant business impact through AI/ML solutions, as demonstrated by quantifiable achievements (e.g., cutting manual review time by 70%, generating incremental revenue, reducing cloud spend).",
        "Strong command of core programming and data technologies (Python, PyTorch, TensorFlow, SQL, Do

In [43]:
%%writefile interview.py

from gemini import ask_json


def generate_questions(resume_json, job_description):

    prompt = f"""
You are a senior technical interviewer.

Generate 10 interview questions based on the resume and job description.

Return ONLY valid JSON.

{{
    "questions":[]
}}

Resume:
{resume_json}

Job Description:
{job_description}
"""

    return ask_json(prompt)


def evaluate_answer(question, answer):

    prompt = f"""
You are a senior technical interviewer.

Evaluate the candidate's answer.

Return ONLY valid JSON.

{{
    "score":0,
    "feedback":"",
    "ideal_answer":""
}}

Question:
{question}

Candidate Answer:
{answer}
"""

    return ask_json(prompt)

Overwriting interview.py


In [44]:
from interview import generate_questions

questions = generate_questions(resume_json, job)

for i, q in enumerate(questions["questions"], 1):
    print(f"\nQuestion {i}")
    print("-"*60)
    print(q)


Question 1
------------------------------------------------------------
{'question': 'Given your extensive experience with Python and machine learning, could you walk me through the typical lifecycle of an ML model from data ingestion to a basic deployment, highlighting key steps an intern would be involved in or need to understand?'}

Question 2
------------------------------------------------------------
{'question': "You've worked with both PyTorch and TensorFlow for advanced projects. For an intern starting out, how would you describe the core differences between these frameworks, and when might one be preferred over the other for a simple deep learning task like image classification?"}

Question 3
------------------------------------------------------------
{'question': 'Your resume mentions extensive data ingestion and transformation with Delta Lake and PySpark. For an intern, what are the fundamental SQL concepts you believe are most critical for querying and manipulating data 

In [45]:
import importlib
import interview

importlib.reload(interview)

print(dir(interview))

['__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'ask_json', 'evaluate_answer', 'generate_questions']


In [46]:
from interview import evaluate_answer
import json

result = evaluate_answer(
    questions["questions"][0],
    "I used PyTorch for image segmentation and optimized the model using Adam optimizer."
)

print(json.dumps(result, indent=4))

{
    "score": 0,
    "feedback": "The candidate's answer is very narrow and does not address the core question about the typical ML model lifecycle from data ingestion to basic deployment. It focuses on a specific task and optimization method (image segmentation with PyTorch and Adam optimizer) rather than outlining the end-to-end process. The answer also fails to highlight key steps an intern would be involved in or need to understand, which was a critical part of the prompt.",
    "ideal_answer": "An ideal answer would outline the end-to-end lifecycle of an ML model, breaking it down into distinct phases and highlighting potential intern involvement:\n\n1.  **Data Ingestion & Collection:** Sourcing, gathering, and storing raw data (e.g., from databases, APIs, web scraping, sensors). *Interns might be involved in data labeling, basic data collection scripts, or identifying data sources.*\n2.  **Data Preprocessing & Cleaning:** Handling missing values, outliers, data type conversion, 

In [47]:
%%writefile app.py

import streamlit as st
import json

from resume_parser import extract_text
from resume_to_json import parse_resume
from ats import ats_score
from interview import generate_questions, evaluate_answer

st.set_page_config(page_title="AI Interview Assistant")

st.title("🤖 AI Interview Assistant")

resume_file = st.file_uploader("Upload Resume (PDF)", type=["pdf"])

job = st.text_area("Paste Job Description")

if resume_file and job:

    with open("temp_resume.pdf", "wb") as f:
        f.write(resume_file.read())

    resume = extract_text("temp_resume.pdf")

    resume_json = parse_resume(resume)

    st.subheader("Resume JSON")
    st.json(resume_json)

    result = ats_score(resume_json, job)

    st.subheader("ATS Score")

    st.metric("Score", f"{result['score']}%")

    st.write("### Matched Skills")
    st.write(result["matched_skills"])

    st.write("### Missing Skills")
    st.write(result["missing_skills"])

    if st.button("Generate Interview Questions"):

        questions = generate_questions(resume_json, job)

        st.session_state.questions = questions["questions"]


if "questions" in st.session_state:

    st.header("Interview")

    for i, q in enumerate(st.session_state.questions):

        st.write(f"### Question {i+1}")

        st.write(q)

        ans = st.text_area(
            f"Answer {i}",
            key=f"ans{i}"
        )

        if st.button(
            f"Evaluate {i}",
            key=f"btn{i}"
        ):

            evaluation = evaluate_answer(q, ans)

            st.success(
                f"Score : {evaluation['score']}/10"
            )

            st.write("Feedback")

            st.write(evaluation["feedback"])

            st.write("Ideal Answer")

            st.write(evaluation["ideal_answer"])

Writing app.py
